# The real-time test: forecasting with the data as it was first published

Every backtest so far used **today's** data. But public-health data is **revised after publication**: the number first reported for a week is usually lower than the final one. CDC's FluSight teams forecast from the **first-published** numbers, so comparing our models (trained on revised data) with theirs was tilted in our favour.

CDC's FluSight hub keeps every past version of its data, tagged with the date it was published (`as_of`). This notebook re-runs the backtest using, for each forecast date, **only the version that existed on that date**, which is exactly what the professionals had, and repeats the comparison under fair conditions.

1. How much are early reports revised?
2. The as-of backtest: SARIMA and the pooled flu-season model (notebook 05)
3. The fair comparison with CDC's ensemble and baseline
4. Can correcting for late reporting help?

## 0. Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

import io, urllib.request
import pandas as pd, numpy as np
from scipy.stats import norm
from statsmodels.tsa.statespace.sarimax import SARIMAX

HUB = 'https://raw.githubusercontent.com/cdcepi/FluSight-forecast-hub/main/'

# every published version of the data (~90 MB, one download)
ts = pd.read_csv(HUB + 'target-data/time-series.csv', parse_dates=['as_of', 'target_end_date'])
print('rows before cleaning:', len(ts))
ts = ts[ts['target'] == 'wk inc flu hosp']
dupes = ts.duplicated(['as_of', 'location', 'target_end_date']).sum()
ts = ts.drop_duplicates(['as_of', 'location', 'target_end_date'])     # one version (2026-05-27) repeats every row
ts['last_week'] = ts.groupby('as_of')['target_end_date'].transform('max')
ts['lag'] = (ts['last_week'] - ts['target_end_date']).dt.days // 7     # weeks since that week was first reportable
print(ts['as_of'].nunique(), 'published versions,', ts['as_of'].min().date(), 'to', ts['as_of'].max().date(),
      '|', dupes, 'duplicate rows dropped')

# today's final numbers: what forecasts are scored against (as CDC does)
final = pd.read_csv(HUB + 'target-data/target-hospital-admissions.csv', parse_dates=['date'])
y_final = final[final.location == 'US'].set_index('date')['value'].sort_index().asfreq('W-SAT')

# FluSight ensemble and baseline forecasts, 23 percentiles
parts = []
for m in ['FluSight-ensemble', 'FluSight-baseline']:
    for r in pd.date_range('2024-08-03', '2026-06-06', freq='W-SAT'):
        try:
            data = urllib.request.urlopen(HUB + f"model-output/{m}/{r.strftime('%Y-%m-%d')}-{m}.csv").read()
        except Exception:
            continue
        f = pd.read_csv(io.BytesIO(data), dtype={'location': str, 'output_type_id': str})
        f = f[(f['location'] == 'US') & (f['target'] == 'wk inc flu hosp') & (f['output_type'] == 'quantile')]
        f['model'] = m
        parts.append(f)
fs = pd.concat(parts)
fs['cutoff'] = pd.to_datetime(fs['reference_date']) - pd.Timedelta(weeks=1)
fs['horizon'] = fs['horizon'] + 1
fs = fs[fs['horizon'].between(1, 4)]
fs['tau'] = fs['output_type_id'].astype(float)
TAUS = sorted(fs['tau'].unique())

rows before cleaning: 1155041


98 published versions, 2023-09-23 to 2026-10-07 | 11925 duplicate rows dropped


## 1. How much are early reports revised?

Example: the week ending 4 Jan 2025, nationally, in successive versions.

In [2]:
us = ts[ts['location'] == 'US']
us[us['target_end_date'] == '2025-01-04'][['as_of', 'observation']].iloc[[0, 1, 2, 4, -1]]

,as_of,observation
218147,2025-01-11,38220.0
226309,2025-01-25,36379.0
234577,2025-02-01,38069.0
251272,2025-02-15,38106.0
1139297,2026-10-07,39626.0


In [3]:
# reported / final, by weeks since first reported, flu season (Nov-Apr), counts >= 50
fin = ts[ts['as_of'] == ts['as_of'].max()][['location', 'target_end_date', 'observation']].rename(columns={'observation': 'final'})
x = ts[ts['lag'].between(0, 3)].merge(fin, on=['location', 'target_end_date'])
x = x[(x['final'] >= 50) & x['target_end_date'].dt.month.isin([11, 12, 1, 2, 3, 4])]
x['ratio'] = x['observation'] / x['final']
print('Median reported/final, by weeks since first reported:')
print(pd.DataFrame({'US': x[x['location'] == 'US'].groupby('lag')['ratio'].median(),
                    'states': x[x['location'] != 'US'].groupby('lag')['ratio'].median()}).round(3))
u0 = x[(x['location'] == 'US') & (x['lag'] == 0)]
print('\nUS newest week, median reported/final by season:')
print(u0.groupby(np.where(u0['target_end_date'].dt.month >= 7, u0['target_end_date'].dt.year,
                          u0['target_end_date'].dt.year - 1))['ratio'].median().round(3))

Median reported/final, by weeks since first reported:
        US  states
lag               
0    0.911   0.952
1    0.963   0.994
2    0.970   1.000
3    0.971   1.000

US newest week, median reported/final by season:
2023    0.970
2024    0.893
2025    0.884
Name: ratio, dtype: float64


**What it shows.**
- **The newest week is under-reported by ~9% nationally** (states ~5%), and fills in over the next 1-2 weeks. The US stays ~3% low for longer: some hospitals report very late.
- **Always in the same direction**, but **growing over time**: ~3% low in 2023-24, ~11% in 2024-25, ~12% in 2025-26.
- This is what a forecaster really starts from, and why a backtest on revised data flatters a model.

## 2. The as-of backtest

For each forecast date, use **the first published version that includes that week**: the data a forecaster had that day. 51 of FluSight's 55 dates have one; the other 4 fall on gaps in CDC's archive (e.g. Thanksgiving week 2025), so every model is compared on the **same 51 dates**.

Both models are unchanged from notebooks 03 and 05: SARIMA (1,1,1)(1,0,0,52), and the pooled model (fourth-root rates across 53 locations, 6 weeks of changes, trained on November-April, range from pooled past errors). Each is refit at every date on that date's version. Forecasts are scored against today's final data.

In [4]:
last_week = ts.groupby('as_of')['target_end_date'].max()
vintage_of = {}
for c in sorted(fs['cutoff'].unique()):
    v = last_week[last_week == c]
    if len(v):
        vintage_of[c] = v.index.min()            # first publication that includes week c
cutoffs = sorted(vintage_of)
print(len(cutoffs), 'of', fs['cutoff'].nunique(), 'forecast dates have a first-published version')

def pinball(a, q, t):
    g = a - q
    return np.where(g > 0, t * g, (t - 1) * g)

def fit_sarima(s):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return SARIMAX(s, order=(1,1,1), seasonal_order=(1,0,0,52),
                       enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

K = 6
cols = [f'chg{k}' for k in range(K)] + ['level']
FLU = [11, 12, 1, 2, 3, 4]

def build_train(rate):
    z = rate ** 0.25
    feat = {f'chg{k}': z.diff().shift(k) for k in range(K)}
    feat['level'] = z
    tabs = []
    for h in range(1, 5):
        tb = pd.concat({n: f.stack() for n, f in feat.items()}, axis=1)
        tb['target'] = (z.shift(-h) - z).stack()
        tb['horizon'] = h
        tabs.append(tb)
    tr = pd.concat(tabs).reset_index().rename(columns={'level_1': 'location_name'})
    tr['target_date'] = tr['date'] + pd.to_timedelta(tr['horizon'], unit='W')
    return tr

def forecast_from(v, c, label_suffix):
    """Fit SARIMA and the pooled flu-season model on one data version v, forecast 1-4 weeks after c."""
    y_v = v[v['location'] == 'US'].set_index('target_end_date')['observation'].sort_index().asfreq('W-SAT')
    rate = (v.pivot_table(index='target_end_date', columns='location_name', values='weekly_rate')
              .sort_index().asfreq('W-SAT').rename_axis('date'))
    scale = (y_v / rate['US']).median()
    f = fit_sarima(y_v[:c]).get_forecast(4)
    mu, se = f.predicted_mean.values, f.se_mean.values
    tr_all = build_train(rate)
    tr_fit = tr_all[tr_all['date'].dt.month.isin(FLU)]
    out = []
    for h in range(1, 5):
        q_s = [max(mu[h-1] + norm.ppf(t) * se[h-1], 0) for t in TAUS]
        tr = tr_fit[(tr_fit['horizon'] == h) & (tr_fit['target_date'] <= c)].dropna()
        X = np.c_[tr[cols].values, np.ones(len(tr))]
        beta = np.linalg.lstsq(X, tr['target'].values, rcond=None)[0]
        err = tr['target'].values - X @ beta
        now = tr_all[(tr_all['horizon'] == h) & (tr_all['location_name'] == 'US') & (tr_all['date'] == c)]
        zhat = now['level'].iloc[0] + np.r_[now[cols].values[0], 1.0] @ beta
        q_p = [max(zhat + np.quantile(err, t), 0) ** 4 * scale for t in TAUS]
        for name, q in [('SARIMA', q_s), ('pooled-flu', q_p)]:
            out += [{'model': f'{name} {label_suffix}', 'cutoff': c, 'horizon': h, 'tau': t, 'value': val}
                    for t, val in zip(TAUS, q)]
    return out

final_v = final.rename(columns={'date': 'target_end_date', 'value': 'observation'})
rows = []
for c in cutoffs:
    rows += forecast_from(ts[ts['as_of'] == vintage_of[c]], c, 'as-of')      # data as first published
    rows += forecast_from(final_v, c, 'revised')                              # today's data, for comparison
ours = pd.DataFrame(rows)
print(ours['model'].unique())

51 of 55 forecast dates have a first-published version


<StringArray>
['SARIMA as-of', 'pooled-flu as-of', 'SARIMA revised', 'pooled-flu revised']
Length: 4, dtype: str


## 3. Results

In [5]:
allq = pd.concat([ours, fs[fs['cutoff'].isin(cutoffs)][['model', 'cutoff', 'horizon', 'tau', 'value']]])
allq['actual'] = [y_final[c + pd.Timedelta(weeks=h)] for c, h in zip(allq['cutoff'], allq['horizon'])]
allq['loss'] = pinball(allq['actual'], allq['value'], allq['tau'])
wis = (allq.groupby(['model', 'cutoff', 'horizon'])['loss'].mean() * 2).rename('wis').reset_index()
w = wis.pivot_table(index='horizon', columns='model', values='wis')
order = ['FluSight-ensemble', 'pooled-flu as-of', 'pooled-flu revised', 'SARIMA as-of', 'SARIMA revised', 'FluSight-baseline']
print('Relative WIS (÷ FluSight-baseline; below 1 is better), 51 dates:')
print(w[order].div(w['FluSight-baseline'], axis=0).round(2).to_string())

Relative WIS (÷ FluSight-baseline; below 1 is better), 51 dates:
model    FluSight-ensemble  pooled-flu as-of  pooled-flu revised  SARIMA as-of  SARIMA revised  FluSight-baseline
horizon                                                                                                          
1.0                   0.70              0.95                0.79          0.79            0.69                1.0
2.0                   0.64              0.81                0.74          0.84            0.97                1.0
3.0                   0.61              0.70                0.67          0.89            1.02                1.0
4.0                   0.59              0.63                0.60          0.89            1.04                1.0


In [6]:
def cost(actual, capacity, ratio=3):
    return np.maximum(capacity - actual, 0).sum() + ratio * np.maximum(actual - capacity, 0).sum()

h4 = (allq[(allq['horizon'] == 4) & allq['tau'].isin([0.1, 0.5, 0.75, 0.9])]
        .pivot_table(index=['model', 'cutoff', 'actual'], columns='tau', values='value').reset_index())
summary = h4.groupby('model').apply(lambda g: pd.Series({
    'MAE': (g[0.5] - g['actual']).abs().mean(),
    'rel WIS': w.loc[4, g.name] / w.loc[4, 'FluSight-baseline'],
    'inside 80%': ((g['actual'] >= g[0.1]) & (g['actual'] <= g[0.9])).mean(),
    'above q90': (g['actual'] > g[0.9]).mean(),
    'best staffing': min(cost(g['actual'], g[t]) for t in [0.5, 0.75, 0.9]),
}))
print('4 weeks ahead, 51 dates:')
print(summary.loc[order].round({'MAE': 0, 'rel WIS': 2, 'inside 80%': 2, 'above q90': 2, 'best staffing': 0}).to_string())

4 weeks ahead, 51 dates:
                       MAE  rel WIS  inside 80%  above q90  best staffing
model                                                                    
FluSight-ensemble   5888.0     0.59        0.78       0.18       618770.0
pooled-flu as-of    6967.0     0.63        0.82       0.16       613675.0
pooled-flu revised  6621.0     0.60        0.84       0.12       646498.0
SARIMA as-of        8105.0     0.89        0.59       0.22       804532.0
SARIMA revised      8986.0     1.04        0.63       0.16       798945.0
FluSight-baseline   8940.0     1.00        0.59       0.20       867060.0


### Does the staffing result depend on the assumed 3:1 cost ratio?

The 3:1 ratio (a week of being short costs three times a week of being overstaffed) was assumed. Sourced wage figures suggest a range:

| How a shortfall is filled | Short : over | Staff to percentile |
|---|---|---|
| Overtime at 1.5× pay | 0.5 : 1 | 33rd |
| Agency nurse, normal market (~1.7× wage) | 0.7 : 1 | 41st |
| Agency nurse, 2022 peak (~3.4× wage) | 2.4 : 1 | 71st |
| Assumed so far | 3 : 1 | 75th |

The cost of being short is only the **premium** over a planned shift, while an idle planned shift costs the full wage. So on wages alone, ratios above 1:1 occur only when agency prices spike; higher ratios mean counting non-wage costs (missed care, burnout, patients diverted).

The rule is fixed in advance and uses no hindsight: at each ratio r, every model staffs to its own percentile r/(1+r) (the nearest of the 23 FluSight percentiles), and the cost is summed over the 51 dates.

In [ ]:
ratios = {'overtime 0.5:1': 0.5, 'agency 0.7:1': 0.7, 'agency peak 2.4:1': 2.4, 'assumed 3:1': 3}
taus = np.array(TAUS)
h4all = allq[allq['horizon'] == 4]
res = {}
for label, r in ratios.items():
    t = taus[np.abs(taus - r / (1 + r)).argmin()]
    g = h4all[np.isclose(h4all['tau'], t)]
    res[f'{label} (q{round(t * 100)})'] = g.groupby('model').apply(lambda d: cost(d['actual'], d['value'], r))
sens = pd.DataFrame(res).loc[order]
print('Staffing cost, 4 weeks ahead, 51 dates (cost points; lower is better):')
print(sens.round(-3).astype(int).to_string())
print('\nAs a share of the FluSight baseline\'s cost:')
print((sens / sens.loc['FluSight-baseline']).round(2).to_string())

**What it shows.**
- **Under the same real-time conditions as the professionals, the pooled model scores a relative WIS of 0.63 at 4 weeks: 37% better than CDC's baseline and within 0.04 of CDC's multi-team ensemble (0.59).**
- **On the staffing decision it ties the ensemble** (best plan ~614k vs ~619k cost points), and its 80% range is more honest (82% inside vs 78%; above q90 16% vs 18%).
- **Real-time data costs the pooled model a little**: relative WIS 0.60 → 0.63 and MAE ~+5% versus revised data on the same dates. That is the honest price of starting from under-reported numbers.
- **SARIMA gets *better* with real-time data at 4 weeks (1.04 → 0.89)**, but worse at 1 week. Not a virtue: SARIMA's main failure is extrapolating momentum, and an under-reported latest week shrinks the recent jump, which happens to dampen its overshoots. An error in the data partly cancelled an error in the model.
- **Which dates you score on matters:** on the 55-date set in notebook 05 the pooled model and the ensemble both scored 0.64; on these 51 dates, with revised data, 0.60 vs 0.59. Every comparison here uses the same 51 dates.

## 4. Can correcting for late reporting help?

Since the newest weeks are consistently under-reported, one natural fix is to scale them up before fitting (a simple **nowcast**). Declared in advance: for each location and weeks-since-first-reported (0-3), use the median ratio of first-reported to later-reported values, learned **only from versions published by the forecast date** and settled weeks (≥ 6 weeks old), and divide the newest weeks by it. Success = 4-week relative WIS better than 0.63 and best staffing cost below 614k, without the range becoming less honest.

In [7]:
def revision_factors(vdate, c):
    past = ts[(ts['as_of'] <= vdate) & ts['lag'].between(0, 3)]
    cur = ts[ts['as_of'] == vdate][['location_name', 'target_end_date', 'observation']].rename(columns={'observation': 'now'})
    cur = cur[cur['target_end_date'] <= c - pd.Timedelta(weeks=6)]
    m = past.merge(cur, on=['location_name', 'target_end_date'])
    m = m[m['now'] >= 20]
    m['ratio'] = m['observation'] / m['now']
    f = m.groupby(['location_name', 'lag'])['ratio'].agg(['median', 'size'])
    return f[f['size'] >= 5]['median']

rows = []
for c in cutoffs:
    v = ts[ts['as_of'] == vintage_of[c]].copy()
    fac = revision_factors(vintage_of[c], c)
    v['factor'] = [fac.get((l, g), 1.0) if g <= 3 else 1.0 for l, g in zip(v['location_name'], v['lag'])]
    v['observation'] /= v['factor']
    v['weekly_rate'] /= v['factor']
    rows += [r for r in forecast_from(v, c, 'as-of + nowcast') if r['model'].startswith('pooled')]
nc = pd.DataFrame(rows)
nc['actual'] = [y_final[c + pd.Timedelta(weeks=h)] for c, h in zip(nc['cutoff'], nc['horizon'])]
nc['loss'] = pinball(nc['actual'], nc['value'], nc['tau'])
w_nc = (nc.groupby(['cutoff', 'horizon'])['loss'].mean() * 2).groupby('horizon').mean()
print('Relative WIS by horizon, pooled-flu:')
print(pd.DataFrame({'as-of': w['pooled-flu as-of'] / w['FluSight-baseline'],
                    'as-of + nowcast': w_nc / w['FluSight-baseline']}).round(3))
g = nc[nc['horizon'] == 4].pivot_table(index=['cutoff', 'actual'], columns='tau', values='value').reset_index()
print('\n4 weeks: best staffing', f"{min(cost(g['actual'], g[t]) for t in [0.5, 0.75, 0.9]):,.0f}",
      '| above q90', round((g['actual'] > g[0.9]).mean(), 2))

Relative WIS by horizon, pooled-flu:
         as-of  as-of + nowcast
horizon                        
1.0      0.946            0.905
2.0      0.809            0.798
3.0      0.704            0.700
4.0      0.630            0.629

4 weeks: best staffing 636,009 | above q90 0.14


**What it shows.** Correcting for late reporting **improves 1-week forecasts** (relative WIS 0.95 → 0.90), where the starting level is most of the forecast. At **4 weeks it makes no meaningful difference** (0.630 → 0.629), and the best staffing plan does not improve (~636k vs ~614k), so by the declared criteria the correction is not adopted. The lesson: the 4-week gap to the professionals is driven by **the turns of the season** (misses of 50-200% at peaks), not by a few percent of late reporting. A correction learned from past seasons also under-corrected while under-reporting was growing.

## 5. Summary

| 4 weeks ahead, 51 dates, real-time data | Relative WIS | Best staffing cost |
|---|---|---|
| FluSight ensemble (dozens of teams) | 0.59 | ~619k |
| **Pooled flu-season model (this project)** | **0.63** | **~614k** |
| SARIMA | 0.89 | ~805k |
| FluSight baseline | 1.00 | ~867k |

**Under the same real-time conditions as CDC's FluSight teams, a single linear model pooled across 53 locations scores 37% better than CDC's baseline, comes within 0.04 of the professional ensemble on CDC's own metric, and ties it on the staffing decision, with a more honest uncertainty range.** Initial reports understate admissions by ~9% nationally (and increasingly so); correcting for that sharpens 1-week forecasts, while the 4-week gap is set by the season's turns.